# Day 25 — Practice SOLUTIONS
### Binomial & Poisson · Module 2 · Topic 2.5

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it tests | Count |
|---|---|---|
| **A** | Predict the output | 8 |
| **B** | Spot & fix the bug | 4 |
| **C** | Write the code | 5 |
| **D** | Challenge | 2 |

**How to work:** for Part A, write your answer down *before* running anything. For every
numeric answer, say what it means in words — *"0.185"* is not an answer, *"about one hour in
five"* is.

Two questions to keep asking yourself:
- Does this say **"out of"** (binomial) or **"per"** (Poisson)?
- Is this *exactly*, *at most*, or *at least*? (and if at least k, is it `1 - cdf(k-1)`?)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from math import comb, factorial, exp
import warnings
warnings.simplefilter("ignore")

rng = np.random.default_rng(25)
print("ready")

---
# Part A — Predict the output

Write your answer first. Then run the cell.

### A1

```python
from math import comb
print([comb(4, k) for k in range(5)])
```

What is printed, and what do those five numbers mean?

In [ ]:
from math import comb
print([comb(4, k) for k in range(5)])

**`[1, 4, 6, 4, 1]`**

These are the numbers of ways to get 0, 1, 2, 3, 4 heads in four coin flips. There is one way
to get zero heads (TTTT), four ways to get exactly one (the head can be in any of four
positions), six ways to get two, and so on.

They add to 16 = 2⁴, which is the total number of sequences — as they must, because every
sequence has *some* number of heads.

In [ ]:
print("sum =", sum(comb(4, k) for k in range(5)), " = 2**4 =", 2**4)
print("as probabilities:", [comb(4,k)/16 for k in range(5)])
print("scipy          :", stats.binom.pmf(range(5), 4, 0.5).round(4))

### A2

```python
print(round(stats.binom.pmf(0, 20, 0.05), 4))
print(round(stats.binom.cdf(0, 20, 0.05), 4))
```

Will these two be the same or different? Why?

In [ ]:
print(round(stats.binom.pmf(0, 20, 0.05), 4))
print(round(stats.binom.cdf(0, 20, 0.05), 4))

**Both print `0.3585` — they are the same.**

`pmf(0)` is "exactly 0". `cdf(0)` is "at most 0", which means 0 and nothing below it — and
there is nothing below 0, because counts cannot be negative.

`k = 0` is the one place where pmf and cdf agree. Everywhere else `cdf(k)` is strictly larger,
because it has more bars in it.

In [ ]:
for k in range(4):
    print(f"k={k}  pmf {stats.binom.pmf(k,20,.05):.4f}   cdf {stats.binom.cdf(k,20,.05):.4f}")

### A3

A quality inspector takes 20 items from a line with a 5% defect rate.

```python
a = stats.binom.cdf(1, 20, 0.05)
b = 1 - stats.binom.cdf(1, 20, 0.05)
print(round(a + b, 6))
```

What prints, and what does that tell you?

In [ ]:
a = stats.binom.cdf(1, 20, 0.05)
b = 1 - stats.binom.cdf(1, 20, 0.05)
print(round(a + b, 6))

**`1.0`**

`a` is "at most 1 defective" and `b` is "2 or more". Between them they cover every possible
outcome with no overlap, so they must sum to exactly 1.

This is a useful check whenever you compute a tail: the thing you want and its complement
should add to 1. If they do not, you have an off-by-one somewhere.

In [ ]:
print(f"at most 1 = {a:.4f}")
print(f"at least 2 = {b:.4f}")
print("complementary pair, so they must sum to 1 ->", round(a+b, 10))

### A4

```python
lam = 3
print(round(stats.poisson.pmf(2, lam), 4))
print(round(stats.poisson.pmf(3, lam), 4))
```

Will the second be bigger, smaller, or equal to the first?

In [ ]:
lam = 3
print(round(stats.poisson.pmf(2, lam), 4))
print(round(stats.poisson.pmf(3, lam), 4))

**Equal — both `0.224`.**

This surprises people. When λ is a whole number, `P(λ-1)` and `P(λ)` are always identical. The
distribution has two joint peaks rather than one.

You can see why from the formula: `P(k)/P(k-1) = λ/k`. When `k = λ` that ratio is exactly 1,
so the two bars match.

In [ ]:
for lm in [1, 2, 3, 7]:
    print(f"lambda={lm}:  P({lm-1})={stats.poisson.pmf(lm-1,lm):.5f}   P({lm})={stats.poisson.pmf(lm,lm):.5f}")
print()
print("ratio P(k)/P(k-1) = lambda/k, which is 1 exactly when k = lambda")

### A5

A shop averages **6 customers per hour**. What is λ for a **10-minute** window?

```python
print(stats.poisson.pmf(0, ???))
```

Work out λ first, then predict the probability of no customers in 10 minutes.

In [ ]:
lam_10min = 6 * (10 / 60)
print("lambda for 10 minutes =", lam_10min)
print("P(no customers) =", round(stats.poisson.pmf(0, lam_10min), 4))

**λ = 1.0, and `P(0) = 0.3679`.**

6 per hour is 1 per 10 minutes. Note that `P(0) = e^(-1) = 0.3679` — with λ=1, a bit over a
third of the windows are empty.

A very common error is to use λ = 6 regardless of the window. That would give
`P(0) = 0.0025` — out by a factor of 148.

In [ ]:
print("correct  (lambda=1) :", round(stats.poisson.pmf(0, 1), 4))
print("wrong    (lambda=6) :", round(stats.poisson.pmf(0, 6), 4))
print("out by a factor of  :", round(stats.poisson.pmf(0,1)/stats.poisson.pmf(0,6), 1))
print()
print("e^-1 =", round(exp(-1), 4))

### A6

```python
d = stats.poisson.rvs(mu=4, size=100000, random_state=1)
print(round(d.mean(), 2), round(d.var(), 2))
```

What two numbers come out, roughly, and why are they related?

In [ ]:
d = stats.poisson.rvs(mu=4, size=100000, random_state=1)
print(round(d.mean(), 2), round(d.var(), 2))

**Both roughly `4.0`.**

For a Poisson, **mean = variance = λ**. No other common distribution does this, and it is why
`variance / mean` is a free diagnostic: a ratio near 1 says Poisson is plausible, a ratio well
above 1 says the events are clustering.

In [ ]:
print("ratio var/mean =", round(d.var()/d.mean(), 3), "-> near 1, Poisson is plausible")
print()
# compare with a binomial, where variance is always SMALLER than the mean
b = stats.binom.rvs(20, 0.2, size=100000, random_state=1)
print(f"binomial(20, 0.2): mean {b.mean():.2f}  var {b.var():.2f}  ratio {b.var()/b.mean():.3f}")
print("binomial variance = np(1-p) < np = mean, always. Underdispersed relative to Poisson.")

### A7

```python
print(round(stats.binom.pmf(2, 1000, 0.003), 6))
print(round(stats.poisson.pmf(2, 3),          6))
```

How close will these two be?

In [ ]:
print(round(stats.binom.pmf(2, 1000, 0.003), 6))
print(round(stats.poisson.pmf(2, 3),          6))

**`0.224154` and `0.224042` — they agree to three decimal places.**

n is large (1000) and p is tiny (0.003), with `n × p = 3`. That is exactly the regime where the
binomial *becomes* the Poisson. The two are not two separate ideas; Poisson is the limit of the
binomial as n grows and p shrinks with the mean held fixed.

In [ ]:
print(f"{'n':>7} {'p':>8} {'binomial':>11} {'Poisson(3)':>12} {'difference':>12}")
for n_, p_ in [(10,.3), (100,.03), (1000,.003), (100000,.00003)]:
    b = stats.binom.pmf(2, n_, p_); po = stats.poisson.pmf(2, 3)
    print(f"{n_:>7} {p_:>8} {b:>11.6f} {po:>12.6f} {abs(b-po):>12.6f}")

### A8

```python
print(stats.binom.pmf(3.5, 10, 0.5))
```

Does this raise an error, return 0, or return something else?

In [ ]:
print(stats.binom.pmf(3.5, 10, 0.5))

**It returns `0.0` — silently, with no error.**

This is the dangerous one. "Exactly 3.5 successes" is a malformed question — you cannot have
half a success — but scipy does not complain. It just returns zero.

So if a probability comes back as exactly `0.0`, check whether you passed a non-integer `k`
before you conclude the event is impossible. A common source is `k = n/2` in Python 3, where
`/` gives a float: `10/2` is `5.0`, which happens to work, but `9/2` is `4.5`, which does not.

In [ ]:
print("pmf(3.5, 10, .5) =", stats.binom.pmf(3.5, 10, .5), " <- silent zero")
print("pmf(3,   10, .5) =", stats.binom.pmf(3,   10, .5))
print()
n = 9
print(f"n/2    = {n/2}  -> pmf = {stats.binom.pmf(n/2, n, .5)}   (float, silently zero)")
print(f"n//2   = {n//2}  -> pmf = {stats.binom.pmf(n//2, n, .5):.4f}  (integer division, correct)")

---
# Part B — Spot & fix the bug

Each cell runs without error and gives a **wrong** answer. Find the bug, say what the wrong
number means, and write the fix.

### B1 — the classic

A factory ships boxes of 50 components, 2% of which are faulty. A box is rejected if it
contains **3 or more** faulty components.

In [ ]:
n, p = 50, 0.02
reject_rate = 1 - stats.binom.cdf(3, n, p)
print(f"Boxes rejected: {reject_rate:.2%}")

**Bug:** `1 - cdf(3)` is "4 or more", not "3 or more". `cdf(3)` already includes the bar at 3,
so subtracting it throws away the very outcome we wanted.

**The wrong number means** the rate of boxes with 4+ faults.

**Fix:** `1 - stats.binom.cdf(2, n, p)` — the rule is `P(at least k) = 1 - cdf(k-1)`.

In [ ]:
n, p = 50, 0.02
wrong = 1 - stats.binom.cdf(3, n, p)
right = 1 - stats.binom.cdf(2, n, p)
print(f"wrong (4 or more): {wrong:.4f}  = {wrong:.2%}")
print(f"right (3 or more): {right:.4f}  = {right:.2%}")
print(f"the bug understates the reject rate by a factor of {right/wrong:.1f}")
print()
print(f"check: P(exactly 3) = {stats.binom.pmf(3, n, p):.4f}")
print(f"       wrong + P(3) = {wrong + stats.binom.pmf(3, n, p):.4f}  == right")

### B2 — the interval

A server logs an average of **12 errors per day**. An on-call engineer wants the probability
of **more than 2 errors during a 4-hour shift**.

In [ ]:
lam = 12
p_busy = 1 - stats.poisson.cdf(2, lam)
print(f"P(more than 2 errors this shift) = {p_busy:.4f}")

**Bug:** λ = 12 is the rate for a *whole day*, but the question asks about a *4-hour shift*.

**The wrong number** (0.9995) says a busy shift is almost certain — it is really answering
"more than 2 errors in a day", which of course is nearly guaranteed when you expect 12.

**Fix:** scale λ to the interval. 4 hours is 1/6 of a day, so λ = 12 × 4/24 = **2**.

In [ ]:
lam_day   = 12
lam_shift = 12 * 4 / 24
wrong = 1 - stats.poisson.cdf(2, lam_day)
right = 1 - stats.poisson.cdf(2, lam_shift)
print(f"lambda for a day   = {lam_day}   -> {wrong:.4f}")
print(f"lambda for 4 hours = {lam_shift}    -> {right:.4f}")
print()
print(f"The bug overstates the risk by {wrong/right:.1f}x.")
print(f"Expected errors per shift: {lam_shift}")

### B3 — the wrong distribution

A pack of 52 cards. Draw 5 **without** replacement. What is the chance of exactly 2 aces?

In [ ]:
print(f"P(exactly 2 aces) = {stats.binom.pmf(2, 5, 4/52):.4f}")

**Bug:** the binomial requires `p` to be the same on every trial. Drawing without replacement
changes it — the first draw has 4/52, the second 4/51 or 3/51 depending on what came out.
**Condition 3 fails.**

**Fix:** use the **hypergeometric** distribution, which is built for sampling without
replacement: `stats.hypergeom.pmf(k, M=52, n=4, N=5)` where M is the population, n the number
of "successes" in it (4 aces), and N the sample size.

The binomial is off by about **17%** here, which is already enough to matter. And it fails
completely in the limit: draw all 52 cards and you are *certain* to have exactly 4 aces, so
P(2 aces) must be 0. The hypergeometric knows that; the binomial still reports a positive
probability.

In [ ]:
binom_ans = stats.binom.pmf(2, 5, 4/52)
hyper_ans = stats.hypergeom.pmf(2, M=52, n=4, N=5)
print(f"binomial (wrong)      : {binom_ans:.6f}")
print(f"hypergeometric (right): {hyper_ans:.6f}")
print(f"relative error        : {abs(binom_ans-hyper_ans)/hyper_ans:.1%}")
print()
print("The binomial does not know the deck runs out. Draw ALL 52 cards:")
print(f"  you must hold exactly 4 aces, so P(2 aces) = 0")
print(f"  hypergeometric: {stats.hypergeom.pmf(2, 52, 4, 52):.4f}   <- correct")
print(f"  binomial      : {stats.binom.pmf(2, 52, 4/52):.4f}   <- impossible, but reported anyway")

### B4 — the model that does not fit

A marketing team logs page views per minute and wants to model them as Poisson so they can
size their servers.

In [ ]:
rng_b4 = np.random.default_rng(404)
# real traffic: quiet most of the time, with bursts when a link gets shared
views = np.concatenate([rng_b4.poisson(2, 1400), rng_b4.poisson(45, 100)])

lam_hat = views.mean()
capacity = stats.poisson.ppf(0.999, lam_hat)
print(f"Fitted lambda = {lam_hat:.2f}")
print(f"Poisson says {capacity:.0f} views/min covers 999 minutes out of every 1000.")
print("-> size the servers for that and we are safe.")

**Bug:** nobody checked whether the data is actually Poisson before fitting one.

Run `views.var() / views.mean()`. For a true Poisson it should be near 1; here it is far
above, because the traffic arrives in **bursts** — which breaks the independence condition.

**The wrong number** is the capacity. The fitted Poisson says 13 views/minute covers
999 minutes in 1,000. In the real data, **6.7%** of minutes exceed 13 — 67 in every 1,000, not
1 — because 100 of those 1,500 minutes came from a burst averaging 45. Size the servers on
this advice and they fall over the first time a link gets shared.

**Fix:** check `var/mean` *before* fitting; if it is well above 1, the data is **overdispersed**
and Poisson is the wrong model. Use the empirical quantiles instead, or a negative binomial.

In [ ]:
rng_b4 = np.random.default_rng(404)
views = np.concatenate([rng_b4.poisson(2, 1400), rng_b4.poisson(45, 100)])

ratio = views.var() / views.mean()
print(f"mean {views.mean():.2f}   var {views.var():.2f}   ratio {ratio:.1f}")
print("ratio >> 1  ->  OVERDISPERSED, Poisson is the wrong model")
print()
cap = stats.poisson.ppf(0.999, views.mean())
model_says = 1 - stats.poisson.cdf(cap, views.mean())
reality    = (views > cap).mean()
print(f"Model capacity (its own 99.9th pct) = {cap:.0f} views/min")
print(f"Poisson says P(> {cap:.0f}/min) = {model_says:.4f}  -> 1 minute in {1/model_says:.0f}")
print(f"The data says           = {reality:.4f}  -> 1 minute in {1/reality:.0f}"
      f"   ({(views>cap).sum()} of {len(views)} minutes)")
print()
print("Empirical quantiles are the honest answer here:")
for q in [50, 95, 99, 99.9]:
    print(f"  {q:>5}th percentile: {np.percentile(views, q):6.1f} views/min")

---
# Part C — Write the code

Write the code yourself. Then say the answer in plain English.

### C1 — an email campaign

You send a campaign to **800** recipients. Historically **4%** click through.

1. What is the expected number of clicks, and the standard deviation?
2. What is the probability of **exactly 32** clicks?
3. What is the probability of **fewer than 25** clicks?
4. What is the probability of **40 or more** clicks?

Be careful with 3 and 4 — "fewer than" and "or more" are different from "at most" and "more
than".

In [ ]:
n, p = 800, 0.04

mean = n * p
sd   = np.sqrt(n * p * (1 - p))
print(f"1.  expected clicks = {mean:.1f}   sd = {sd:.2f}")

print(f"2.  P(exactly 32)   = {stats.binom.pmf(32, n, p):.4f}")

#  'fewer than 25' means 0..24, so cdf(24)
print(f"3.  P(fewer than 25)= {stats.binom.cdf(24, n, p):.4f}")

#  '40 or more' means 1 - cdf(39)
print(f"4.  P(40 or more)   = {1 - stats.binom.cdf(39, n, p):.4f}")

print()
print(f"In words: expect about {mean:.0f} clicks, give or take {sd:.0f}.")
print(f"A campaign under 25 clicks happens {stats.binom.cdf(24,n,p):.1%} of the time -")
print("so a single weak campaign is not yet evidence that anything is wrong.")

**Expected 32 clicks, sd 5.54.**

- P(exactly 32) = **0.0718**
- P(fewer than 25) = **0.0837** — note this is `cdf(24)`, not `cdf(25)`
- P(40 or more) = **0.0912** — `1 - cdf(39)`, not `1 - cdf(40)`

In words: expect about 32 clicks, give or take 6. Roughly 8% of campaigns come in under 25
and roughly 9% come in at 40 or above, *with no change at all in the underlying click rate*.
That is the practical point — one weak campaign is not evidence that something broke.

### C2 — a hospital ward

A ward admits an average of **5 emergency patients per night**.

1. What is the probability of a completely quiet night (zero admissions)?
2. What is the probability of **at most 3**?
3. The ward has 8 emergency beds. What fraction of nights do they run out?
4. Plot the distribution and shade the nights where they run out.

In [ ]:
lam = 5

print(f"1.  P(zero admissions)  = {stats.poisson.pmf(0, lam):.4f}")
print(f"2.  P(at most 3)        = {stats.poisson.cdf(3, lam):.4f}")

#  'run out' means MORE than 8 arrive, i.e. 9 or more
run_out = 1 - stats.poisson.cdf(8, lam)
print(f"3.  P(more than 8)      = {run_out:.4f}  -> {run_out*365:.0f} nights a year")

ks = np.arange(0, 16)
pr = stats.poisson.pmf(ks, lam)
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(ks, pr, color=["#2FA98C" if k <= 8 else "#E8674C" for k in ks])
ax.set_xticks(ks)
ax.set_xlabel("Emergency admissions in one night"); ax.set_ylabel("Probability")
ax.set_title(f"Poisson(5): red = beds run out, {run_out:.1%} of nights")
plt.show()

- P(zero) = **0.0067** — a completely quiet night happens about twice a year
- P(at most 3) = **0.2650**
- P(more than 8) = **0.0681** → about **25 nights a year** the ward runs out of beds

The subtle bit is question 3. Eight beds means they cope with 8; they run out when **9 or
more** arrive, which is `1 - cdf(8)`, not `1 - cdf(9)`. Using `cdf(9)` would give 3.2% and
halve the estimate.

### C3 — which distribution?

For each of the following, say **binomial or Poisson**, name the parameters, and compute the
asked-for probability.

1. 15 students sit a test with a 70% pass rate. P(all 15 pass)?
2. A proofreader finds on average 2.5 typos per page. P(a page with no typos)?
3. 1 in 200 manufactured chips is faulty. In a tray of 400, P(at most 1 faulty)?
4. A junction sees 1.2 accidents a month. P(3 or more next month)?

In [ ]:
print("1. BINOMIAL - '15 students' gives a countable n")
print(f"   n=15, p=0.70   P(all pass) = {stats.binom.pmf(15, 15, 0.70):.5f}")
print()
print("2. POISSON - 'per page' is a rate, there is no n")
print(f"   lambda=2.5     P(no typos) = {stats.poisson.pmf(0, 2.5):.4f}")
print()
print("3. BINOMIAL - 'a tray of 400' gives n (Poisson(2) is a close approximation)")
b = stats.binom.cdf(1, 400, 1/200)
print(f"   n=400, p=0.005 P(at most 1) = {b:.4f}")
print(f"   Poisson(2) approximation    = {stats.poisson.cdf(1, 2):.4f}")
print()
print("4. POISSON - 'per month' is a rate")
print(f"   lambda=1.2     P(3 or more) = {1 - stats.poisson.cdf(2, 1.2):.4f}")

1. **Binomial**, n=15, p=0.70 → `pmf(15, 15, 0.7)` = **0.00475**. Less than a 1-in-200 chance
   that all fifteen pass, even with a 70% pass rate.
2. **Poisson**, λ=2.5 → `pmf(0, 2.5)` = **0.0821**
3. **Binomial**, n=400, p=0.005 → `cdf(1, 400, 0.005)` = **0.4053**. Poisson(λ=2) gives 0.4060
   — close, because n is large and p is small, exactly the convergence regime.
4. **Poisson**, λ=1.2 → `1 - cdf(2, 1.2)` = **0.1205**

The tell works every time: "15 students", "a tray of 400" → *out of*, so binomial. "per page",
"a month" → *per*, so Poisson.

### C4 — simulate it

Do not trust a formula you have not tested.

1. Simulate 100,000 batches of 20 items at a 5% defect rate (use `rng.binomial`).
2. Compare the simulated proportion with 0, 1, 2 and 3 defects against `binom.pmf`.
3. Compare the simulated mean and sd against `np` and `sqrt(np(1-p))`.
4. Do the same for Poisson(λ=3) against `rng.poisson`.

In [ ]:
rng_c4 = np.random.default_rng(2504)
n, p = 20, 0.05

sim = rng_c4.binomial(n, p, size=100_000)

print(f"{'k':>3} {'simulated':>11} {'formula':>11} {'diff':>9}")
for k in range(4):
    s = (sim == k).mean()
    f = stats.binom.pmf(k, n, p)
    print(f"{k:>3} {s:>11.4f} {f:>11.4f} {abs(s-f):>9.4f}")

print()
print(f"mean  simulated {sim.mean():.4f}   formula {n*p:.4f}")
print(f"sd    simulated {sim.std():.4f}   formula {np.sqrt(n*p*(1-p)):.4f}")

print()
lam = 3
sim_p = rng_c4.poisson(lam, size=100_000)
print(f"{'k':>3} {'simulated':>11} {'formula':>11}")
for k in range(4):
    print(f"{k:>3} {(sim_p==k).mean():>11.4f} {stats.poisson.pmf(k,lam):>11.4f}")
print()
print(f"Poisson mean {sim_p.mean():.3f}  var {sim_p.var():.3f}  <- both near lambda = {lam}")

Simulation and formula agree to about three decimal places everywhere, which is what 100,000
draws buys you.

Two things worth taking from this:

- The formulas are not magic. They are counting arguments, and a simulation reproduces them.
- When you meet a situation whose distribution you *cannot* name, you can still simulate it.
  That is the fallback for every messy real problem, and it is how Day 27 will make the
  Central Limit Theorem visible.

### C5 — a real decision

An online shop gets **40 orders an hour** on an ordinary day. Each order is packed by hand and
one packer can handle **45 an hour**.

1. With one packer, what fraction of hours does the backlog grow?
2. How many packers are needed so that happens less than **1 hour per 40-hour week**?
3. Express the answer in hours per week, not probabilities.
4. Why would staffing to the average of 40 be a mistake?

In [ ]:
lam = 40
week = 40  # hours

for packers in range(1, 4):
    cap = 45 * packers
    #  backlog grows when MORE than cap orders arrive
    risk = 1 - stats.poisson.cdf(cap, lam)
    print(f"{packers} packer(s), capacity {cap:>3}/hr:  "
          f"P(backlog grows) = {risk:.4f}  ->  {risk*week:5.2f} hours/week")

print()
print(f"Standard deviation of hourly orders = {np.sqrt(lam):.1f}")
print(f"A 'normal' busy hour is {lam + 2*np.sqrt(lam):.0f} orders - above one packer's 45.")

- **One packer**: `1 - poisson.cdf(45, 40)` = **0.1903**, so the backlog grows in about 19% of
  hours — **7.6 hours a week**.
- **Two packers** (capacity 90): essentially zero. One packer is not enough and two is ample;
  there is no in-between because capacity jumps in units of 45.
- Stated usefully: *"with one packer we fall behind for about 7.6 hours of a 40-hour week."*

Why the average misleads: 40 orders against a capacity of 45 looks like 12% headroom. But the
standard deviation is `sqrt(40)` = 6.3, so a perfectly ordinary busy hour brings 40 + 2(6.3) ≈
**53 orders** — well past what one packer can do. The average describes the typical hour; the
distribution describes the hours that hurt, and those are the ones that decide staffing.

---
# Part D — Challenge

### D1 — the A/B test that is not evidence

Your team runs an A/B test on a checkout button.

- Version A: 1,000 visitors, **48** conversions
- Version B: 1,000 visitors, **62** conversions

B is 29% better. The team wants to ship it.

**Your job:** assume the true rate is identical for both and equal to the pooled rate. Under
that assumption, how often would you see a gap of 14 or more conversions *purely by chance*?

Simulate it — 100,000 pairs of binomial draws — and give the team a one-sentence verdict.

*Hint: use `rng.binomial(1000, pooled_rate, size=100_000)` twice and compare.*

In [ ]:
rng_d1 = np.random.default_rng(2501)

n_each = 1000
a_obs, b_obs = 48, 62
pooled = (a_obs + b_obs) / (2 * n_each)
print(f"pooled conversion rate = {pooled:.4f}")
print(f"observed gap           = {b_obs - a_obs} conversions")
print()

#  both versions are truly identical - simulate the null world
sim_a = rng_d1.binomial(n_each, pooled, size=100_000)
sim_b = rng_d1.binomial(n_each, pooled, size=100_000)
gap = sim_b - sim_a

p_one_sided = (gap >= (b_obs - a_obs)).mean()
p_two_sided = (np.abs(gap) >= (b_obs - a_obs)).mean()

print(f"P(gap >= 14 by chance alone)      = {p_one_sided:.4f}")
print(f"P(|gap| >= 14 by chance alone)    = {p_two_sided:.4f}")
print()
print(f"sd of the gap under the null = {gap.std():.2f} conversions")
print(f"the observed gap is {(b_obs-a_obs)/gap.std():.2f} standard deviations out")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(gap, bins=np.arange(gap.min(), gap.max()+2)-0.5, color="#D8E1EB")
ax.hist(gap[np.abs(gap) >= 14], bins=np.arange(gap.min(), gap.max()+2)-0.5, color="#E8674C")
ax.axvline(14, color="#152238", ls="--", lw=2)
ax.set_xlabel("B conversions minus A conversions, if the two are identical")
ax.set_ylabel("simulated trials")
ax.set_title("The observed gap of 14 is not unusual in a world where nothing differs")
plt.show()

**Verdict for the team:**

> *"If the two buttons converted at exactly the same rate, we would still see a gap this large
> or larger about 1 time in 5. This test does not show that B is better — it shows that 1,000
> visitors is not enough to tell."*

The numbers: the gap has a standard deviation of about 10 conversions under the null, so the
observed 14 is **1.37 standard deviations** out — comfortably inside the range of ordinary
noise. The two-sided chance is **18.6%**.

Note what the binomial bought us here. "29% better" sounds decisive because it is a ratio of
two small counts. Modelling the counts tells you how much those counts wobble on their own, and
48 vs 62 is well inside the wobble.

**What would settle it?** Run the simulation again with the same rates but more visitors per
arm. At 1,000 per arm this test would catch a real 4.8%-vs-6.2% difference only **26%** of the
time; at 10,000 per arm, **99%** of the time. That calculation is how you choose a sample size
*before* you run the test, not after.

In [ ]:
#  how big a test would settle it?
true_a, true_b = 0.048, 0.062
for n_each in [1_000, 5_000, 10_000, 20_000]:
    sa = rng_d1.binomial(n_each, true_a, size=20_000)
    sb = rng_d1.binomial(n_each, true_b, size=20_000)
    #  under the null (both at the pooled rate) how big a gap is 'surprising'?
    na = rng_d1.binomial(n_each, (true_a+true_b)/2, size=20_000)
    nb = rng_d1.binomial(n_each, (true_a+true_b)/2, size=20_000)
    threshold = np.percentile(np.abs(nb - na), 95)
    power = (np.abs(sb - sa) > threshold).mean()
    print(f"n={n_each:>6} per arm:  detect the real difference {power:>5.1%} of the time")

### D2 — when Poisson breaks

Below are two real-looking count datasets. One is genuinely Poisson; the other is not.

1. For each, compute the mean, the variance and the ratio.
2. Plot each against the Poisson pmf with λ = the sample mean.
3. Say which one fails, **which condition it breaks**, and what you would use instead.
4. For the failing one, compare what Poisson predicts for the 99th percentile against what the
   data actually shows — and say what that would cost someone who trusted the model.

In [ ]:
rng_d2 = np.random.default_rng(2502)

#  dataset 1: support tickets per hour
tickets = rng_d2.poisson(6, size=2000)

#  dataset 2: goals per football match (from a league with a few very high-scoring teams)
goals = np.concatenate([rng_d2.poisson(1.2, 1800), rng_d2.poisson(11, 200)])

print("tickets:", tickets[:15])
print("goals  :", goals[:15])

In [ ]:
for name, d in [("tickets", tickets), ("goals", goals)]:
    ratio = d.var() / d.mean()
    verdict = "Poisson plausible" if ratio < 1.5 else "OVERDISPERSED"
    print(f"{name:<9} mean {d.mean():6.2f}   var {d.var():7.2f}   ratio {ratio:6.2f}   {verdict}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, (name, d) in zip(axes, [("tickets", tickets), ("goals", goals)]):
    hi = d.max() + 1
    ks = np.arange(hi)
    observed = np.array([(d == k).mean() for k in ks])
    ax.bar(ks, observed, color="#3E7CB1", label="observed")
    ax.plot(ks, stats.poisson.pmf(ks, d.mean()), color="#E8674C", lw=2.5,
            marker="o", ms=3, label=f"Poisson({d.mean():.1f})")
    ax.set_title(f"{name}  -  var/mean = {d.var()/d.mean():.2f}")
    ax.set_xlabel("count"); ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("proportion")
plt.tight_layout(); plt.show()

In [ ]:
#  what the failing model would cost you
d = goals
lam_hat = d.mean()
model_p99 = stats.poisson.ppf(0.99, lam_hat)
data_p99  = np.percentile(d, 99)

print(f"fitted lambda           = {lam_hat:.2f}")
print(f"Poisson says 99th pct   = {model_p99:.0f} goals")
print(f"the data says 99th pct  = {data_p99:.0f} goals")
print()
over = (d > model_p99).mean()
print(f"Poisson expects 1% of matches above {model_p99:.0f}.")
print(f"In reality {over:.1%} of matches are - {over/0.01:.0f} times as many.")

**1 & 2.** `tickets` has `var/mean ≈ 1` and the orange Poisson curve lies on top of the bars.
`goals` has `var/mean` far above 1 and the curve misses badly: it over-predicts the middle and
collapses to nothing in the right tail, where the data still has mass.

**3. `goals` fails.** The giveaway is overdispersion, and the condition it breaks is
**constant rate**: the dataset mixes two populations — 1,800 ordinary matches around λ=1.2 and
200 high-scoring ones around λ=11. There is no single λ that describes both, so the fitted mean
(about 2.2) describes *neither*. It sits above the typical match and far below a high-scoring
one.

What to use instead: a **negative binomial**, which has a second parameter precisely for extra
spread; or, better, split the data and model the two groups separately — if you can identify
which is which, that is more informative than any single-distribution fit. If you only need a
tail number and not a model, the empirical quantiles are honest and require no assumption at
all.

**4. The cost.** Poisson puts the 99th percentile at **6 goals** and says only 1% of matches go
above it. In the data, **9.3%** of matches do — nine times as many, and the real 99th
percentile is **16**. Anyone pricing an over/under-6-goals bet off this model, or sizing
capacity for a "1-in-100" event, would be wrong by an order of magnitude *in the tail* — which
is exactly the part of the distribution these models get used for.

**The general lesson:** `variance / mean` takes one line and catches this before it reaches
production. Run it on any count data before you fit a Poisson to it.

---
# Where the marks were

| Question | The thing being tested |
|---|---|
| A1, A2 | pmf vs cdf, and the one place they agree |
| A3 | complementary events must sum to 1 |
| A4 | the twin peaks when λ is a whole number |
| A5 | **scaling λ to the interval** |
| A6 | mean = variance = λ |
| A7 | binomial → Poisson convergence |
| A8 | the silent zero from a non-integer k |
| B1 | ⚠️ `1 - cdf(k-1)`, not `1 - cdf(k)` |
| B2 | ⚠️ λ must match the window |
| B3 | p must not change — without replacement breaks it |
| B4 | check `var/mean` **before** fitting |
| C1–C3 | wording: *exactly / at most / fewer than / at least* |
| C4 | simulation as a check on every formula |
| C5 | translating a probability into a staffing decision |
| D1 | a difference in counts is not evidence until you model the noise |
| D2 | diagnosing a model that does not fit, and pricing the error |

If you only remember two things from today:

1. **"At least k" is `1 - cdf(k-1)`.** Every other mistake on this sheet announces itself;
   this one does not.
2. **An average cannot tell you how often you will be overwhelmed.** A distribution can. That
   is what the whole topic is for.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*